# TrustRank and Spam Detection

## Learning Objectives

1. **Explain** how spam farms manipulate PageRank
2. **Implement** TrustRank as biased PageRank from a trusted seed set
3. **Define** topic-sensitive PageRank and its personalization vector
4. **Compute** the spam mass of a page

## Spam Farms

A **spam farm** is a link structure designed to artificially boost the PageRank of a target page $t$.

**Structure:**
- Many "supporting pages" $s_1, \ldots, s_k$ each link to $t$
- $t$ links back to all $s_i$ (distributing PageRank to them)
- External pages $x_1, \ldots$ link to $t$ (real in-links)

**Effect:** each supporting page receives rank from $t$ and sends it back — creating a multiplier effect.

**Rank of $t$** with $k$ supporting pages and external in-flow $e$:

$$r_t \approx \frac{\beta e}{1-\beta^2} + \frac{k \beta^2}{n(1-\beta^2)}$$

Even small $e$ gets amplified significantly with large $k$.

In [1]:
from collections import defaultdict

def pagerank_teleportation(out_edges, nodes, beta=0.85, n_iter=100, teleport_to=None):
    """PageRank with optional non-uniform teleportation vector."""
    n = len(nodes); out_deg = {u: len(out_edges.get(u,[])) for u in nodes}
    if teleport_to is None:
        teleport_weights = {v: 1.0/n for v in nodes}
    else:
        s = sum(teleport_to.values())
        teleport_weights = {v: teleport_to.get(v,0)/s for v in nodes}
    r = {v: 1.0/n for v in nodes}
    for _ in range(n_iter):
        new_r = {v: 0.0 for v in nodes}
        dangling = sum(r[u] for u in nodes if out_deg[u]==0)
        for u in nodes:
            if out_deg[u]>0:
                for v in out_edges.get(u,[]):
                    new_r[v] += beta * r[u] / out_deg[u]
        for v in nodes:
            new_r[v] += (beta*dangling + (1-beta)) * teleport_weights[v]
        r = new_r
    return r

# Build spam farm example
nodes = ['legit1','legit2','target','spam1','spam2','spam3']
out_e = {
    'legit1': ['legit2','target'],
    'legit2': ['legit1'],
    'target': ['spam1','spam2','spam3'],
    'spam1':  ['target'],
    'spam2':  ['target'],
    'spam3':  ['target'],
}
r_normal = pagerank_teleportation(out_e, nodes, beta=0.85)
print("PageRank with spam farm:")
for v,rank in sorted(r_normal.items(), key=lambda x:-x[1]):
    bar='#'*int(rank*50)
    print(f"  {v:10s}: {rank:.4f}  {bar}")

PageRank with spam farm:
  target    : 0.4307  #####################
  spam1     : 0.1470  #######
  spam2     : 0.1470  #######
  spam3     : 0.1470  #######
  legit1    : 0.0724  ###
  legit2    : 0.0558  ##


## TrustRank

**Idea:** start the teleportation bias from a **trusted seed set** (known-good pages, manually curated).

Trust propagates along links but decays:
- A page linked to by a trusted page inherits some trust
- Pages reachable only through many hops receive little trust
- Spam pages (linked to by other spam) receive near-zero trust

$$\text{TrustRank}(v) = \text{PageRank with teleportation} \to \text{seed set only}$$

**Spam mass:** fraction of page's PageRank that comes from spam:
$$\text{SpamMass}(v) = \frac{r(v) - \text{TrustRank}(v)}{r(v)}$$

In [2]:
# TrustRank from legitimate seed nodes
seed = {'legit1': 0.5, 'legit2': 0.5}
r_trust = pagerank_teleportation(out_e, nodes, beta=0.85, teleport_to=seed)

print("TrustRank (teleport only to legit pages):")
for v,trust in sorted(r_trust.items(), key=lambda x:-x[1]):
    normal = r_normal[v]
    spam_mass = (normal - trust) / normal if normal > 0 else 0
    bar = '#'*int(spam_mass*30)
    print(f"  {v:10s}: TrustRank={trust:.4f}  SpamMass={spam_mass:.3f}  {bar}")

TrustRank (teleport only to legit pages):
  target    : TrustRank=0.3327  SpamMass=0.228  ######
  legit1    : TrustRank=0.2172  SpamMass=-2.000  
  legit2    : TrustRank=0.1673  SpamMass=-2.000  
  spam1     : TrustRank=0.0943  SpamMass=0.359  ##########
  spam2     : TrustRank=0.0943  SpamMass=0.359  ##########
  spam3     : TrustRank=0.0943  SpamMass=0.359  ##########


## Topic-Sensitive PageRank

Standard PageRank has a single global ranking.
**Topic-sensitive PageRank** computes a different ranking for each topic $T_i$:

$$\text{PageRank}_{T_i}(v) = \text{PageRank with teleportation} \to T_i \text{ seed pages only}$$

**Usage:**
1. Pre-compute $k$ PageRank vectors (one per topic, e.g., sports, health, technology)
2. For a given query, estimate user's topic interest vector $\mathbf{q}$
3. Rank pages by $\sum_i q_i \cdot \text{PageRank}_{T_i}(v)$

This gives personalized rankings without recomputing PageRank per user.

In [3]:
# Two-topic example
nodes_t = ['sports1','sports2','news1','tech1','tech2','shared']
out_t = {
    'sports1':['sports2','shared'], 'sports2':['sports1'],
    'news1':['sports1','tech1'],
    'tech1':['tech2','shared'],     'tech2':['tech1'],
    'shared':['sports1','tech1'],
}

seed_sports = {'sports1':0.7,'sports2':0.3}
seed_tech   = {'tech1':0.7,'tech2':0.3}

r_sports = pagerank_teleportation(out_t, nodes_t, beta=0.85, teleport_to=seed_sports)
r_tech   = pagerank_teleportation(out_t, nodes_t, beta=0.85, teleport_to=seed_tech)

print(f"{'Page':>10} {'Sports PR':>12} {'Tech PR':>12}")
for v in nodes_t:
    print(f"{v:>10} {r_sports[v]:>12.4f} {r_tech[v]:>12.4f}")

      Page    Sports PR      Tech PR
   sports1       0.3702       0.1460
   sports2       0.2024       0.0620
     news1       0.0000       0.0000
     tech1       0.1460       0.3702
     tech2       0.0620       0.2024
    shared       0.2194       0.2194


## Summary

| Technique | Teleportation target | Purpose |
|-----------|---------------------|---------|
| Standard PageRank | Uniform over all pages | General authority |
| TrustRank | Trusted seed set | Spam detection |
| Topic-sensitive PR | Topic-specific seed pages | Personalization |

**Spam mass** threshold: pages with SpamMass > 0.8 are likely spam farms.
**Topic-sensitive PR** requires $k$ PageRank computations (offline) + dot product at query time.